In [1]:
# This is to write a private package of Jack polynomials
import itertools
import math
from typing import List, Type
%run Hilbert_C^2_package.ipynb

# justify whether the skew diagram l1/l2 is a horizontal strip
def is_horizontal_strip(self, l2):
    #l1 = Partition(self)
    l1 = self
    # l1 should be either the same length with l2 or one more elements than l2
    jg1 = (len(l1) == len(l2) or len(l1) == len(l2) + 1)
    
    # justify whether two partition are overlacing
    jg2 = all(item[0]>=item[1] for item in zip(l1,l2))
    jg3 = all(item[0]<=item[1] for item in zip(l1[1:],l2))
    return jg1 and jg2 and jg3
Partition.is_horizontal_strip = is_horizontal_strip

# use tree structure to define the weighted tableaux
class TreeNode:
    def __init__(self, value):
        self.value = value
        self.children = []
        self.parent = None
    
    def grow(self, child):
        if isinstance(child, TreeNode):
            self.children.append(child)
            child.parent = self
        else:
            raise TypeError(f'{child} is not a TreeNode')
    
    def next_layer(self, weight: int, outer_bound: Type[Partition]):
        '''
        this function is specific to generate weighted tableau, so self.value need to be a partition.
        
        INPUT:
        
        - ``self`` -- a TreeNode instance with partition as value
        - ``weight`` -- int
        - ``outer_bound `` -- Partition
        
        OUTPUT:
        
        - returns children nodes with their value being partitions containing self.value and the skew diagram is 
          a horizontal strip
        '''
        if isinstance(self.value, Partition):
            dim = sum(self.value) + weight
            gap = Partitions(dim, inner = self.value, outer = outer_bound)
            res = filter(lambda p:p.is_horizontal_strip(self.value), gap)
            treelized_res = [TreeNode(p) for p in res]
            
            for item in treelized_res:
                self.grow(item)
                
            return treelized_res
        else:
            raise TypeError('tree node value is not a partition')
    
    def path_to_root(self):
        path = [self]
        while path[-1].parent != None:
            path.append(path[-1].parent)
        return path[::-1]



In [2]:
def weight_check(func): # ..TO DO :: this filter is waited to be renewed for general cases.
    def wrapper(list1, list2):
        if sum(list1) <= sum(list2):
            return func(list1, list2)
        else:
            raise ValueError("Second partition is not a weight of the first partition.")
    
    return wrapper

def row_slash_column(self, mu):
    '''
    Partition(mu) is a subpartition of Partition(self)
    this function is R_{l\m} - C_{l\m} defined in (10.11) VI, Macdonald's book, 
    serves as the index set of psi below
    
    INPUT:
    - ``self`` -- a partition
    - ``mu`` -- a partition
    
    OUTPUT:
    - returns cells in R_{l\m} - C_{l\m} as a list
    '''
    # cells of l-m
    difference = list(filter(lambda s:s not in mu.cells(),self.cells()))
    
    # rows intersect l-m
    vertical_range = list(set(map(lambda s:s[0], difference)))
    
    # columns intersect l-m
    horizontal_range = list(set(map(lambda s:s[1], difference)))

        #res = list(itertools.product(vertical_range, range(width)))
    res = list(filter(lambda s:s[0] in vertical_range and not s[1] in horizontal_range, self.cells()))
    return res
    
Partition.row_slash_column = row_slash_column

def psi(self, mu):
    '''
    this function is \psi_{l,m}(s) defined in (10.11) VI, Macdonald's book
    
    INPUT:
    - ``self`` -- a partition 
    - ``mu`` -- a partition
    
    OUTPUT:
    - returns a polynomial in Jack polynomial parameter a
    '''
    base_ring.<a> = PolynomialRing(QQ)
    b = lambda mu, s: (a*mu.arm(s) + mu.leg(s) + 1)/(a*mu.arm(s) + mu.leg(s) + a)
    res = prod(map(lambda s : b(mu,s)/b(self,s), self.row_slash_column(mu)))
    return res
Partition.psi = psi

def macdonald_tableau(self, tab):
    '''
    convert standard tableaus to Macdonald's definition as a sequence of partitions
    self is the shape of tableau
    tab is the tableau represented by filling numbers into diagram of self
    
    INPUT:
    
    - ``self`` -- a partition as a list
    - ``tab`` -- a standard tableau as a nested list
    
    OUTPUT:
    
    - returns a list of partitions determined by the standard young tableau tab
    '''

@weight_check
def weighted_tableau(self, weight: List[int]) -> List[Partition]:
    '''
    here we take the definition of Macdonald's to generate a tableau with shape 'self' and weight 'weight'.
    definition is in the first section partiton of his book. 'weight' should be a partition smaller than 'self'
    in lexical order
    
    INPUT:
    
    - ``self`` -- a partition as a list
    - ``weight`` -- a partition as a list
    
    OUTPUT:
    
    - returns a list of partitions satisfying the definition of weighted tableau
    '''
    root = TreeNode(Partition([]))
    cache = [root]
    whole_tree = []
    
    # generate all possibilities as a tree
    for idx in range(len(weight)):
        generate_next_layer = lambda treenode : treenode.next_layer(weight[idx], self)
        next_layer = map(generate_next_layer, cache)
        cache = chain(*next_layer)
        whole_tree.append(cache)
        #print([new_nodes.value for new_nodes in cache])
    
    # a weighted tableau is a path from root to leaf
    all_path = [list(map(lambda node:node.value, leaf.path_to_root())) for leaf in cache]
    
    return all_path
Partition.weighted_tableau = weighted_tableau

def psi_T(self, tab):
    '''
    definition of psi_T is at (10.12) of Ch. VI in I. Macdonald's book
    tab is a weighted tableau of shape self
    
    INPUT:
    - `` self`` -- a partition
    - `` tab`` -- a list of partitions satisfying the definition of weighted tableau
    
    OUTPUT:
    - returns a polynomial in Jack polynomial parameter a
    
    REMARK: 
    maybe it's better that we write a class 'WeightedTableau'
    '''
    base_ring.<a> = PolynomialRing(QQ)
    length = len(tab)
    res = prod([tab[idx].psi(tab[idx - 1]) for idx in range(1, length)])
    return res
Partition.psi_T = psi_T

def u(self, mu):
    '''
    definition of u_{self,mu} is at (10.13) of Ch. VI in I. Macdonald's book.
    u_{self,mu} is the coefficient in expression of jack polynomial JP_{self} as linear combination of
    monomial symmetric functions.
    mu is a partition smaller than self in lexical order.
    
    
    INPUT:
    - ``self`` -- a partition
    - ``mu`` -- a partition as a list
    
    OUTPUT:
    - returns a polynomial in Jack polynomial parameter a
    '''
    base_ring.<a> = PolynomialRing(QQ)
    all_tab = self.weighted_tableau(mu)
    res = sum([self.psi_T(tab) for tab in all_tab])
    return res
Partition.u = u

def E_index(self, mu):
    '''
    Generate the index set for summation in function E_{lambda, mu}. 
    lambda is smaller than mu in lexical order
    
    INPUT:
    - ``self`` -- a partition
    - ``mu`` -- a partition
    
    OUTPUT:
    - returns a filter class
    '''
    is_split_sum = lambda nu : sorted(chain(*nu), reverse = True) == self
    all_nu = itertools.product(*[Partitions(part) for part in mu])
    res = filter(is_split_sum, all_nu)
    return res
Partition.E_index = E_index

def E(self, mu):
    '''
    E_{lambda,mu} has the property: m_lambda = \sum E_{lambda,mu}p_mu/z_mu; p_mu = \sum E_{lambda,mu}h_lambda.
    lambda is smaller than mu in lexical order.
    E_{lambda,mu} = lambda.E(mu)
    
    INPUT:
    - ``self`` -- a partition
    - ``mu`` -- a partition
    
    OUTPUT:
    - returns a rational number
    '''
    E_summand = lambda nu : \
    prod([mu[i] * math.factorial(len(nu[i]) - 1) / nu[i].multiplicity_fac() for i in range(len(mu))])
    
    res = (-1)^(len(self) - len(mu)) * sum(map(E_summand, self.E_index(mu)))
    
    return res
Partition.E = E

def derivative(self, multi):
    '''
     compute the derivative of p_self by p_lambda, where multi is the multiplicity form of partition lambda
    lambda = multiplicity_to_partition(multi)
    lambda should be a subpartition of mu, but we didn't add the filter here.
    
    INPUT:
    - `` self`` -- a partition
    - `` multi`` -- List[int]
    
    OUTPUT:
    - returns a tuple
    '''
    mu_multiplicity = partition_to_multiplicity(self)
    if type(multi) != tuple:
        raise TypeError(f'{multi} should be a tuple consisting of multiplicities')

    extended_multi = list(multi) + (self[0]-len(multi))*[0]
    
    if not all(item[0]>=item[1] for item in zip(mu_multiplicity,extended_multi)):
        return (0,0)
    mu_derivatived = (
        multiplicity_to_partition(
            list(map(
                lambda item:item[0]-item[1], 
                zip(mu_multiplicity,extended_multi)
            ))
        )
    )

    derivative_coefficient = prod(
        map(
            lambda item: prod(range(item[0]-item[1]+1,item[0]+1)),
            zip(mu_multiplicity,extended_multi)
        )
    )
    
    return (derivative_coefficient, mu_derivatived) # d/dp[nu](p[mu]) = derivative_coefficient*p[mu_derivatived]
Partition.derivative = derivative

In [13]:
# Jack polynomial P_lambda^a = m(lambda) + \sum_{part < lambda} lambda.u(part) * m(part)
# m(mu) = \sum_{mu < nu} mu.E(nu)/nu.z() * p(nu)
def sym_setting(func):
    def wrapper(mu):
        dim = sum(mu)
        m = SymmetricFunctions(QQ).m()
        p = SymmetricFunctions(QQ).p()
        R.<a> = PolynomialRing(QQ);
        base_ring = R
        base_field = FractionField(base_ring);
        monomial_mod = CombinatorialFreeModule(base_field, Partitions(dim), prefix='m')
        power_sum_mod = CombinatorialFreeModule(base_field, Partitions(dim), prefix='p')
        return func(mu)
    return wrapper

@sym_setting
def monomial_to_power_sum(mu):
    if isinstance(mu, Partition):
        p = SymmetricFunctions(QQ).p()
        dim = sum(mu)
        return sum([mu.E(nu)/nu.z() * p(nu) for nu in Partitions(dim)])
    else:
        raise TypeError(f'{mu} is not a Partition')

class Jack:
    def __init__(self, mu, parameter = None):
        '''
        Default parameter is a
        
        INPUT:
        - ``mu`` -- a list
        - ``parameter`` -- a rational polynomial or ordinary polynomial
        '''
        if isinstance(Partition(mu), Partition) and type(mu) == list:
            self.mu = mu
            self.dim = sum(mu)
            self.parameter = parameter
        else:
            raise TypeError(f'{mu} is not a partition written as list')
        # rf = rational function
        self.type_rf = sage.rings.fraction_field_element.FractionFieldElement 
        # mp = multivariable polynomial
        self.type_mp = sage.rings.polynomial.multi_polynomial_libsingular.MPolynomial_libsingular 
        # r = rational number
        self.type_r = sage.rings.rational.Rational
        # int = integer
        self.type_int = sage.rings.integer.Integer
        
    def var_tuple(self):
        if self.parameter == None:
            return ('a')
        elif type(self.parameter) == self.type_rf:
            res = self.parameter.numerator().variables() + self.parameter.denominator().variables()
            return tuple(map(lambda x:str(x), res))
        elif type(self.parameter) == self.type_mp:
            res = self.parameter.variables()
            return tuple(map(lambda x:str(x), res))
    
    def var_num(self):
        return len(self.var_tuple())
    
    def var_name(self):
        return ''.join(self.var_tuple())
    
    def base_ring(self):
        return PolynomialRing(QQ, self.var_num(), names = self.var_name())
    
    def base_field(self):
        return FractionField(self.base_ring())
    
    def monomial_mod(self):
        return CombinatorialFreeModule(self.base_field(), Partitions(self.dim), prefix='m')
    
    def power_sum_mod(self):
        return CombinatorialFreeModule(self.base_field(), Partitions(self.dim), prefix='p')

    def _m(self):
        '''
        express jack polynomials as linear sum of monomial symmetric functions
        
        INPUT:
        - `` self`` -- a partition written as a list
        
        OUTPUT:
        - returns an element in self.monomial_mod
        '''

        p_iterator = iter(Partitions(self.dim)) # p stands for partition
        while next(p_iterator) != self.mu:
            pass
        jack_m = (
            sum(map(
                lambda nu:Partition(self.mu).u(nu) * self.monomial_mod()(nu) ,p_iterator
            ))
            + self.monomial_mod()(self.mu)
        )
        return jack_m
    
    def _to_p(self, nu):
        '''
        Express jack polynomials as linear sum of power sum symmetric functions.
        This function gives the coefficient at p_nu. p stands for power sum
        
        INPUT:
        - `` self`` -- a partition written as a list
        - ``nu`` -- a partition
        
        OUTPUT:
        - returns a polynomial in jack parameter a
        '''
        p_iterator = iter(Partitions(self.dim)) # p stands for partition
        while next(p_iterator) != self.mu:
            pass
        coeff_p = (
            sum(map(
                lambda part: Partition(self.mu).u(part) * part.E(nu) / nu.z(),
                p_iterator
            ))
            + Partition(self.mu).E(nu)/nu.z()
        )
        if type(coeff_p) == self.type_r or type(coeff_p) == self.type_int:
            return self.base_ring()(coeff_p)
        else:
            return coeff_p(self.parameter)
        
    def _p(self, mod=lambda nu:1):
        '''
        express jack polynomials as linear sum of power sum symmetric functions. 
        mod is modification on linear coefficients defined as a function on partitions.
        default value of mod is constant 1 on partitions.
        
        INPUT:
        - `` self`` -- a partition written as a list
        - ``mod`` -- lambda function
        
        OUTPUT:
        - returns an element in self.power_sum_mod()
        '''
        if self.parameter != None:
            if self.parameter.parent() != self.base_ring() and self.parameter.parent() != self.base_field():
                raise TypeError(
                    f'base ring {self.parameter.parent()} of {self.parameter} is not the same as {self.base_ring()}'
                )
        
        p_iterator = iter(Partitions(self.dim))
        jack_p = (
            sum(map(
                lambda nu:self._to_p(nu) * mod(nu) * self.power_sum_mod()(nu), p_iterator
            ))
        )
        return jack_p
    
    def modification(self):
        '''
        Modificate Jack polynomial so that the inner product can be aligned with geometric inner product in 
        cohomology of Hilbert scheme.
        
        INPUT:
        - `` self`` -- a partition written as a list
        
        OUTPUT:
        - returns an element in QQ[q,t]
        '''
        if len(self.base_ring().gens()) == 2:
            q = self.base_ring().gen()
            t = self.base_ring().gen(1)
            p = Partition(self.mu)
            
            if self.parameter == -q/t:
                c_iterator = iter(p.cells()) # c stands for cell
                
                mod = (
                    prod(
                        map(lambda s: -(p.leg(s) + 1)*t + p.arm(s)*q, c_iterator)
                    )
                )
                return mod
            else:
                raise ValueError('Jack parameter should be -q/t')
        else:
            raise ValueError('Jack parameter should be -q/t')
    
    def _normalized(self, C=False):
        '''
        Normalize Jack polynomial so that the inner product can be aligned with geometric inner product in 
        cohomology of Hilbert scheme. C = True means we take E. Carlsson's modification p_k = q*p_k
        
        INPUT:
        - `` self`` -- a partition written as a list
        - ``C`` -- Boolean
        
        OUTPUT:
        - returns an element in self.power_sum_mod() with coefficients in QQ[q,t]
        '''
        if C == True:
            normalized_jack = self._p(lambda nu:q^len(nu)) * self.modification()
        else:
            normalized_jack = self._p() * self.modification()

        return normalized_jack
    
    def _normalized_dict(self, C=False):
        '''
        Normalize Jack polynomial so that the inner product can be aligned with geometric inner product in 
        cohomology of Hilbert scheme. C = True means we take E. Carlsson's modification p_k = q*p_k
        
        INPUT:
        - `` self`` -- a partition written as a list
        - ``C`` -- Boolean
        
        OUTPUT:
        - returns a dict with key being partitions and values are coefficient of J_mu at p_key
        '''
        jack_dict = {}
        p_iterator = iter(Partitions(self.dim))
        for part in p_iterator:
            jack_dict[part] = self._to_p(part) * self.modification() * q^(C*len(part))
            
        return jack_dict

    def derivative(self, multi, C = False):
        '''
        compute the derivative of NORMALIZED J_mu by p_lambda, 
        where multi is the multiplicity form of partition lambda
        lambda = multiplicity_to_partition(multi)
        lambda should be a subpartition of mu, but we didn't add the filter here.
        
        INPUT:
        - `` self`` -- a partition
        - `` multi`` -- Tuple[int]
        - `` C`` -- choose whether to use Carlsson's modification p_k*q = p_k
        
        OUTPUT:
        - returns a dict
        
        EXAMPLE:
        
        normalized J_[2,2] = t^4*p[[1, 1, 1, 1]] + (-2*q*t^3-2*t^4)*p[[2, 1, 1]]
        + (q^2*t^2-q*t^3+t^4)*p[[2, 2]] + 4*q*t^3*p[[3, 1]] + (-q^2*t^2-q*t^3)*p[[4]]
        
        Jack([2,2],-q/t).derivative([1]) =
        {0: 0, [3]: 4*q*t^3, [2, 1]: -4*q*t^3 - 4*t^4, [1, 1, 1]: 4*t^4}
        
        Jack([2,2],-q/t).derivative([0,1]) =
        {0: 0, [2]: 2*q^2*t^2 - 2*q*t^3 + 2*t^4, [1, 1]: -2*q*t^3 - 2*t^4}
        
        Jack([2,2],-q/t).derivative([0,0,1]) =
        {0: 0, [1]: 4*q*t^3}
        '''
        derivative_dict = {0:0}
        lam = multiplicity_to_partition(multi)
        q = self.base_ring().gen()
        
        # compute derivative of J_mu by p_lam
        p_iterator = iter(Partitions(self.dim))
        for part in p_iterator:
            derivative_dict[part.derivative(multi)[1]]\
            = q^(C*len(part)) * self._to_p(part) * part.derivative(multi)[0] * self.modification()
            #print(part.derivative(multi))

        return derivative_dict

In [26]:
# check Carlsson's combinatoric formula
 
def Gamma(mu, twist, C=False, sign = True):
    '''
    gamma_dict is the expansion of exp(twist* \sum_n (-1)^{n+1}\frac{\p}{\p_n})J_mu 
    on power sum symmetric functions
    INPUT:
    - ``mu`` -- Partition
    - ``twist`` -- polynomial in QQ[m,q,t]
    - `` C`` -- choose whether to use Carlsson's modification p_k*q = p_k
    
    OUTPUT:
    returns a dict
    '''
    dim = sum(mu)
    mu_multiplicity = partition_to_multiplicity(mu)
    # print('mu_multiplicity:',tuple(mu_multiplicity))
    
    index = chain(*[Partitions(d) for d in range(dim)])
    # it seems that iterator can be called once after defined. so we cannot use list(iter).

    J = Jack(list(mu),-q/t)
    #der_dict = J.derivative(mul) for mul in index
    gamma_dict = {0:0}
    
    # nonconstant terms
    for nu_part in index:
        
        nu_mult = partition_to_multiplicity(nu_part)
        multiplier = (-1)^(sign*(sum(nu_part) - len(nu_part)))/prod(map(math.factorial, nu_mult))\
        * twist^len(nu_part)
        #print('sign:',(-1)^(sign*(sum(nu_part) - len(nu_part))))
        der_dict = J.derivative(tuple(nu_mult),C)
        der_dict = {key : value*multiplier for key,value in der_dict.items()}
        gamma_dict.update(der_dict)
    

    # constant term
    constant = 0
    for nu in iter(Partitions(dim)):
        nu_mult = partition_to_multiplicity(nu)
        #print('sign:',(-1)^(sign*(sum(nu) - len(nu))))
        multiplier = (-1)^(sign*(sum(nu) - len(nu)))/prod(map(math.factorial, nu_mult)) * twist^len(nu)
        adder = J.derivative(nu_mult, C).get(Partition([]))*multiplier
        constant += adder

    gamma_dict[Partition([])] = constant

    return gamma_dict

def scalar_prod(part):
    base.<a> = PolynomialRing(QQ)
    part_mult = partition_to_multiplicity(part)
    
    z_part = prod(map(math.factorial,part_mult)) * prod(part)
    sign = (-1)^(sum(part)-len(part))
    
    res = sign^1 * z_part * a^len(part)
    return res

def sqsum(part1,part2):
    '''
    compute part1\sqcup part2 as partitions
    
    INPUT:
    - ``part1`` - partition
    - ``part2`` - partition
    
    OUTPUT:
    - returns a partition
    '''
    mult1 = partition_to_multiplicity(part1)
    mult2 = partition_to_multiplicity(part2)
    
    disjoint_sum = tuple(a + b for a, b in itertools.zip_longest(mult1, mult2, fillvalue=0))

    res = multiplicity_to_partition(disjoint_sum)
    
    return res

def intersection(part1,part2):
    '''
    compute part1\cap part2 as partitions
    
    INPUT:
    - ``part1`` - partition
    - ``part2`` - partition
    
    OUTPUT:
    - returns a partition
    '''
    res = tuple(min(a, b) for a, b in itertools.zip_longest(part1, part2, fillvalue=0))
    return Partition(res)

def scalar_prod(part, s=True):
    '''
    compute inner product <p_part,p_part>
    
    INPUT:
    - ``part`` - partition
    
    OUTPUT:
    - returns a polynomial in 'a'
    '''
    base.<a> = PolynomialRing(QQ)
    part_mult = partition_to_multiplicity(part)
    z_part = prod(map(math.factorial,part_mult)) * prod(part)
    sign = (-1)^(sum(part)-len(part))
    res = sign^s * z_part * a^len(part)
    return res

In [65]:
import time
base.<q,t> = PolynomialRing(QQ)

lam = Partition([2])
mu = Partition([2])

res = sqsum(mu,lam)
print('sqsum is:',res)
res2 = intersection(mu,lam)
print('intersection is:', res2)

dim = sum(mu)
para = -base.gen()/base.gen(1)

J_lam = Jack(list(lam),para)
J_mu = Jack(list(mu), para)

mu_jack_dict = J_mu._normalized_dict(C = True)
lam_jack_dict = J_lam._normalized_dict(C = True)

print(mu_jack_dict)

twister_base.<m,q,t> = PolynomialRing(QQ)
theta = t
twist1 = m
twist2 = theta - m

expected_res = (-1)^(sum(lam))*theta^(-sum(lam)-sum(mu))\
* prod(map(lambda s: m + lam.arm(s) + 1 + theta*mu.leg(s) , lam.cells()))\
* prod(map(lambda s: m - mu.arm(s) - theta*(lam.leg(s)+1) , mu.cells()))

# all partitions contained in lam\cap mu
index = chain(*[Partitions(d,outer = res2) for d in range(0,dim+1)])
inprod = 0; i = 0
for part in index:
    i+=1
    print(f'\nthe {i}-th loop with kappa being {part}')
    summands_mu = lambda nu:(-1)^(sum(mu)-sum(part))*twist1^(len(nu))\
    *scalar_prod(sqsum(nu,part),s=False)(1/theta)\
    *mu_jack_dict[sqsum(nu,part)]/nu.z()
    prod_mu = (
        sum(
            map(
                summands_mu,
                Partitions(sum(mu)-sum(part))
            )
        )
    )
    print('prod mu:',factor(prod_mu))
    
    summands_lam = lambda nu:(-1)^(sum(lam)-sum(part))*twist2^(len(nu))\
    *scalar_prod(sqsum(nu,part),s=False)(1/theta)\
    *lam_jack_dict[sqsum(nu,part)]/nu.z()
    prod_lam = (
        sum(
            map(
                summands_lam,
                Partitions(sum(lam)-sum(part))
            )
        )
    )
    print('prod lam:',factor(prod_lam))
    
    multiplier = (-1)^(sum(part)-len(part)) * theta^len(part) / part.z()
    print(multiplier)
    
    
    inprod += multiplier * prod_mu * prod_lam
    
print('\nactual result is:',factor(inprod))
print('\nexpected result is:', expected_res)
print('\ndifference is:', factor(expected_res*(t*q)^(sum(lam)+sum(mu)) - inprod))

sqsum is: [2, 2]
intersection is: [2]
{[2]: -q^2*t, [1, 1]: q^2*t^2}

the 1-th loop with kappa being []
prod mu: m * (m - 1) * q^2
prod lam: (-m + t) * (-m + t - 1) * q^2
1

the 2-th loop with kappa being [1]
prod mu: (-2) * m * q^2
prod lam: (2) * (m - t) * q^2
t

the 3-th loop with kappa being [2]
prod mu: (-2) * q^2
prod lam: (-2) * q^2
-1/2*t

actual result is: q^4 * (m^4 - 2*m^3*t + m^2*t^2 - 3*m^2*t + 3*m*t^2 - m^2 + m*t - 2*t)

expected result is: (m^4 - 2*m^3*t + m^2*t^2 + 2*m^3 - 5*m^2*t + 3*m*t^2 - m^2 - m*t + 2*t^2 - 2*m + 2*t)/t^4

difference is: (2) * q^4 * (m^3 - m^2*t - m*t + t^2 - m + 2*t)


In [6]:
mu = Partition([5,3])
J_mu = Jack(list(mu),para)
mu_jack_dict = J_mu._normalized_dict(C=True)

lam = Partition([1,1,1,1,1])
J_lam = Jack(list(lam),para)
lam_jack_dict = J_lam._normalized_dict(C=True)

dim = sum(mu)
H = Hilb(dim)
'''# compute <p_1J_mu,J_lambda>
new_dict = {Partition(key+[1]):value for key, value in mu_jack_dict.items()}
res = sum([new_dict.get(part,0)*lam_jack_dict.get(part)*scalar_prod(part)(1/(q*t)) for part in Partitions(dim+1)])
print(factor(res))'''

# compute <J_mu,J_mu>
res = sum([mu_jack_dict.get(part)^2 * scalar_prod(part)(1/(q*t)) for part in Partitions(sum(mu))])
print(factor(res))
print(factor(H.eu_tangent(mu)))

# compute <J_mu,p1^dim>
p1dim = Partition([1 for x in range(dim)])
res = mu_jack_dict[p1dim]*scalar_prod(p1dim)(1/(q*t))
print(res)

(-48) * (-5*q + t) * (-4*q + t) * (-3*q + t) * (-3*q + 2*t) * t^2 * (-2*q + t)^2 * (-q + t)^3 * q^5
(-48) * (-5*q + t) * (-4*q + t) * (-3*q + t) * (-3*q + 2*t) * t^2 * (-2*q + t)^2 * (-q + t)^3 * q^5
40320


In [7]:
Partition([3])+[1]

[3, 1]

In [8]:
JP = SymmetricFunctions(FractionField(QQ['t'])).jack().P()

exmp = [2]
mu = Partition(exmp)
dim = sum(exmp)
part_list = [1,1]
part = Partition(part_list)
coeff = JP._to_m(mu)

# we need to load _self_to_m firstly, then the function _to_m can call _self_to_m_cache
JP._self_to_m(JP([dim]))

res1 = coeff(part)
print(res1)
res2 = mu.u(part)
print(res2)

2/(t + 1)
2/(a + 1)


In [14]:
# test on Partition.E()
base.<q,t> = PolynomialRing(QQ)
lam_ = [2,2]
lam = Partition(lam_)
mu = Partition([1,1])
res = lam.E(mu)/mu.z()

J = Jack(lam_, -q/t)
jp = Jack(lam_)._p()

print(res)
print(monomial_to_power_sum(lam))
print('\n')
print(f'jack polynomial of {lam} is \n{Jack(lam_)._m()}\n')

print(f'jack polynomial of {lam} is \n{jp}\n')

jn = J._normalized(C=False)
print(f'normalized jack polynomial of {lam} is \n{jn}\n')

#print('\n',*lam.weighted_tableau(mu))
#print(lam.u(mu))

print(J.parameter)
#print('\n',Jack(lam_)._to_p(mu))

0
1/2*p[2, 2] - 1/2*p[4]


jack polynomial of [2, 2] is 
(12/(a^2+3*a+2))*m[[1, 1, 1, 1]] + (2/(a+1))*m[[2, 1, 1]] + m[[2, 2]]

jack polynomial of [2, 2] is 
(1/2/(a^2+3*a+2))*p[[1, 1, 1, 1]] + ((a-1)/(a^2+3*a+2))*p[[2, 1, 1]] + ((1/2*a^2+1/2*a+1/2)/(a^2+3*a+2))*p[[2, 2]] + ((-2*a)/(a^2+3*a+2))*p[[3, 1]] + ((-1/2*a^2+1/2*a)/(a^2+3*a+2))*p[[4]]

normalized jack polynomial of [2, 2] is 
t^4*p[[1, 1, 1, 1]] + (-2*q*t^3-2*t^4)*p[[2, 1, 1]] + (q^2*t^2-q*t^3+t^4)*p[[2, 2]] + 4*q*t^3*p[[3, 1]] + (-q^2*t^2-q*t^3)*p[[4]]

(-q)/t


In [ ]:
from sage.rings.fraction_field_element import is_FractionFieldElement
base_ring.<x,y> = PolynomialRing(QQ)
f = x/y
g = x + y
h = base_ring(1/2)

type_r = sage.rings.rational.Rational
type_int = sage.rings.integer.Integer

var_list = f.numerator().variables() + f.denominator().variables()
test = 'c'
print(len(test))
another_ring = PolynomialRing(QQ, len(test), names=test);
ff = another_ring.gen() + 1
print(another_ring)
print(h.is_constant())
print(ff)

In [10]:
mu = Partition([4,2,1])
lam = Partition([7])
for s in lam.cells():
    res = mu.leg(s)
    print(res)


2
1
0
0
-1
-1
-1
